<p style="margin: 0;">
  <!-- Left logo stays on the left -->
  <img src="./../Images/logo.png" alt="logo" width="280" style="margin-bottom: 0;" />

  <!-- Right-aligned container -->
  <span style="float: right; text-align: right;">
    <img src="./../Images/ai_chemy_logo.png"
         alt="ai_chemy logo"
         width="280"
         style="margin-top: 20px; display: block;" />
    <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">
      <img alt="Creative Commons Licence"
           style="border-width: 0; margin-top: 10px; display: block;"
           src="https://i.creativecommons.org/l/by/4.0/88x31.png"
           title="This work is licensed under a Creative Commons Attribution 4.0 International License." />
    </a>
  </span>
</p>
 
 <h1 style="margin-top: 0; text-align: center;">Session 8: Docking with AutoDock Vina.
</h1>
<hr style="border: 1px solid black;">

This project has received funding from the AI for Chemistry: AIchemy Hub (EPSRC grant EP/Y028775/1 and EP/Y028759/1) and educational support from the <a href="https://github.com/Edinburgh-Chemistry-Teaching/Data-driven-chemistry">Data-Driven Chemistry</a>  course at the University of Edinburgh’s School of Chemistry.

Authors: 

- Dr Alex Aziz
- Mr Zhaohui Jiang

Email: a.aziz@mmu.ac.uk
<hr style="border: 1px solid black;">

<span style="font-weight: normal; font-size: 16px;">

In this final session, you will evaluate how the top predicted SweetLead compounds might physically interact with HIV‑1 protease using molecular docking. Docking is a computational technique that predicts how a ligand fits into a protein’s active site and estimates the binding affinity. While QSAR modelling predicts activity based on molecular descriptors, docking provides a structure‑based perspective by analysing the 3D interactions between the ligand and the target protein.

AutoDock Vina is a widely used, open‑source docking program known for its speed and good accuracy. It searches for favourable ligand poses within the binding pocket and scores them according to predicted binding energy (lower scores indicate stronger predicted binding). By docking the top predicted inhibitors, we can visually inspect how they fit into the HIV‑1 protease active site and assess whether the binding modes are chemically plausible.

This step complements the QSAR predictions by adding structural validation and helps identify the most promising compounds for further analysis or potential repurposing.

</span>

In [ ]:
import os
#You will need to update with your correct path and have AutoDock Vina installed
os.environ["VINA_PYTHON"] = "/path_to_AutoDock_Vina/build/python" #Use the correct path

import sys

# Load AutoDock Vina Python bindings via environment variable
vina_path = os.getenv("VINA_PYTHON")

if vina_path is None:
    raise ValueError(
        "Environment variable VINA_PYTHON is not set.\n"
        "Please set it to your AutoDock Vina python module path, e.g.:\n"
        'export VINA_PYTHON="/path/to/AutoDock-Vina/build/python"'
    )

# Append the user-defined path
sys.path.append(vina_path)

from vina import Vina
from Bio.PDB import PDBParser, PDBIO
import matplotlib.pyplot as plt
import py3Dmol

# Importing the SWEETLEAD DATABASE

In this section, you load the SWEETLEAD database and investigate the binding of indinavir predicted by our Random Forest model to be the most potent HIV‑1 protease inhibitors. These selected molecules will then be prepared for structure‑based docking using AutoDock Vina, which generates 3D conformers and predicts how each compound fits into the HIV‑1 protease active site. Finally, you will visualise the docked poses to assess their binding modes and potential as novel inhibitors.

In [ ]:
# Importing the SWEETLEAD DATABASE
import pandas as pd

txt_file = "./SWEETLEAD_data/SWEETLEAD.txt"
df = pd.read_csv(txt_file, sep="\t")  # or sep="," if comma-separated
df.columns = df.columns.str.strip()   # remove any leading/trailing spaces

# Search for Indinavir in OfficialNames
indinavir = df[df['OfficialNames'].str.contains("indinavir", case=False, na=False)]
indinavir


In [ ]:
# ===============================
# SWEETLEAD Indinavir 5 Conformers
# ===============================

import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import rdmolfiles

# Load SWEETLEAD TXT database
txt_file = "./SWEETLEAD_data/SWEETLEAD.txt"
df = pd.read_csv(txt_file, sep="\t")  # adjust sep="\t" or sep="," if needed
df.columns = df.columns.str.strip()   # clean up column names

# Extract Indinavir row
indinavir_row = df[df['OfficialNames'].str.contains("indinavir", case=False, na=False)]
if indinavir_row.empty:
    raise ValueError("Indinavir not found in SWEETLEAD")

# Retrieve the SMILES code
smiles = indinavir_row.iloc[0]['Isomeric Smiles']

# Create RDKit molecule
mol = Chem.MolFromSmiles(smiles)
if mol is None:
    raise ValueError("Could not parse SMILES")

mol = Chem.AddHs(mol)  # add explicit hydrogens

# Generate conformers
num_conformers = 5
conformer_ids = AllChem.EmbedMultipleConfs(
    mol, numConfs=num_conformers, pruneRmsThresh=0.5, randomSeed=42
)

# Optimize conformers
for conf_id in conformer_ids:
    AllChem.UFFOptimizeMolecule(mol, confId=conf_id)

# Create output directory (named after ligand)
ligand_name = "indinavir"
output_dir = os.path.join(".", ligand_name)
os.makedirs(output_dir, exist_ok=True)

# Save each conformer in its own PDB file
for i, conf_id in enumerate(conformer_ids):
    filename = os.path.join(output_dir, f"{ligand_name}_conf{i+1}.pdb")
    rdmolfiles.MolToPDBFile(mol, filename, confId=conf_id)

print(f" Saved {len(conformer_ids)} conformers inside folder: {output_dir}")

# Save all conformers together in one multi-model PDB
multi_model_file = os.path.join(output_dir, f"{ligand_name}_multi.pdb")
rdmolfiles.MolToPDBFile(mol, multi_model_file, confId=-1)
print(f" Multi-model PDB saved as: {multi_model_file}")



In [ ]:
# Prepare the target

import os
from Bio.PDB import PDBList

# Make a directory for PDB files
protein_directory = "protein_structures"
os.makedirs(protein_directory, exist_ok=True)

# PDB ID you want to download
pdb_id = "4DJQ"

# Initialize PDB downloader
pdbl = PDBList()
pdbl.retrieve_pdb_file(pdb_id, pdir=protein_directory, file_format="pdb")

print(f"{pdb_id} downloaded into {protein_directory}/")


In [ ]:
#Visualising the protein
protein_file = "protein_structures/4DJQ.pdb"

with open(protein_file) as f:
    pdb_data = f.read()

import py3Dmol

# Initialize viewer
view = py3Dmol.view(width=800, height=600)
view.addModel(pdb_data, "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})  # protein cartoon
view.addSurface(py3Dmol.VDW, {"opacity":0.7, "color":"red"})  # optional
view.zoomTo()
view.show()

In [ ]:
from openbabel import pybel
import os

# Paths
receptor_file = "./protein_structures/4DJQ.pdb"
out_dir = "./receptor_pdbqt"
os.makedirs(out_dir, exist_ok=True)
receptor_pdbqt = os.path.join(out_dir, "4DJQ.pdbqt")

# Load receptor
mol = next(pybel.readfile("pdb", receptor_file))
mol.addh()       # add hydrogens
# DO NOT call make3D() for proteins

# Write temporary ligand-style PDBQT
tmp_file = os.path.join(out_dir, "tmp_receptor.pdbqt")
mol.write("pdbqt", tmp_file, overwrite=True)

# Fix receptor PDBQT: remove ROOT/BRANCH lines
with open(tmp_file, "r") as f_in, open(receptor_pdbqt, "w") as f_out:
    for line in f_in:
        if line.startswith(("ROOT", "BRANCH", "TORSDOF")):
            continue  # skip ligand-only tags
        f_out.write(line)

print(" Receptor PDBQT compatible with Vina written to:", receptor_pdbqt)


In [ ]:
from openbabel import pybel
import os

# Paths
receptor_file = "./protein_structures/4DJQ.pdb"
ligand_dir = "./indinavir"
out_dir = "./indinavir_pdbqt"
os.makedirs(out_dir, exist_ok=True)

# --- Convert receptor to PDBQT ---
mol = next(pybel.readfile("pdb", receptor_file))
mol.addh()
mol.make3D()
mol.write("pdbqt", os.path.join(out_dir, "4DJQ.pdbqt"), overwrite=True)
print(" Receptor converted to PDBQT")

# --- Convert ligand conformers to PDBQT ---
for file in os.listdir(ligand_dir):
    if file.endswith(".pdb"):
        lig = next(pybel.readfile("pdb", os.path.join(ligand_dir, file)))
        lig.addh()
        lig.make3D()
        out_name = file.replace(".pdb", ".pdbqt")
        lig.write("pdbqt", os.path.join(out_dir, out_name), overwrite=True)
        print(f" Ligand converted: {out_name}")


In [ ]:
from openbabel import pybel
import numpy as np

ligand_file = "./indinavir_pdbqt/indinavir_conf5.pdbqt"

lig = next(pybel.readfile("pdbqt", ligand_file))
coords = np.array([atom.coords for atom in lig])

# Compute geometric center
center = coords.mean(axis=0)

# Compute size along each axis + padding
size = coords.max(axis=0) - coords.min(axis=0) + 2.0  # 2 Å padding

print("Center:", center)
print("Box size:", size.tolist())


In [ ]:
from vina import Vina
import os

# Paths
receptor_file = "./receptor_pdbqt/4DJQ.pdbqt"
ligand_dir = "./indinavir_pdbqt"
output_dir = "./docking_results"
os.makedirs(output_dir, exist_ok=True)

# Binding site coordinates and box size (adjust as needed)
center = [2, -5, 3]
box_size = [22, 22, 22]

# Store results
results = []

# Loop through all ligand conformers
for ligand_file in sorted(os.listdir(ligand_dir)):
    if ligand_file.endswith(".pdbqt") and "conf" in ligand_file:
        ligand_path = os.path.join(ligand_dir, ligand_file)
        print(f"Docking {ligand_file} ...")
        
        # Initialize Vina
        v = Vina(sf_name='vina')
        v.set_receptor(receptor_file)
        v.set_ligand_from_file(ligand_path)
        v.compute_vina_maps(center=center, box_size=box_size)
        
        # Optional: score initial pose
        energy_before = v.score()[0]
        
        # Optional: local minimization
        energy_min = v.optimize()[0]
        
        # Dock ligand
        v.dock(exhaustiveness=32, n_poses=20)
        
        # Save top poses (e.g., top 5)
        out_file = os.path.join(output_dir, ligand_file.replace(".pdbqt", "_out.pdbqt"))
        v.write_poses(out_file, n_poses=5, overwrite=True)
        
        # Save best affinity
        best_energy = min([e[0] for e in v.energies()])
        results.append({
            "Ligand": ligand_file,
            "Best_affinity_kcal/mol": best_energy
        })

# Save results as CSV
import pandas as pd
df = pd.DataFrame(results)
df.to_csv(os.path.join(output_dir, "vina_docking_results.csv"), index=False)
df


In [ ]:
# Read ligand PDBQT
with open('./docking_results/indinavir_conf2_out.pdbqt') as f:
    pdbqt_str = f.read()

# Keep only PDB-like lines (ATOM/HETATM)
pdb_str = "\n".join([line for line in pdbqt_str.splitlines() if line.startswith(("ATOM", "HETATM"))])

import py3Dmol
view = py3Dmol.view(width=500, height=500)
view.addModel(pdb_str, 'pdb')   # <-- load as pdb now
view.setStyle({'stick': {}})
view.zoomTo()
view.show()


In [ ]:
# Receptor PDBQT → clean to PDB
with open('./receptor_pdbqt/4DJQ.pdbqt') as f:
    rec_str = f.read()
rec_pdb = "\n".join([l for l in rec_str.splitlines() if l.startswith(("ATOM","HETATM"))])

# Ligand PDBQT → clean to PDB
with open('./docking_results/indinavir_conf3_out.pdbqt') as f:
    lig_str = f.read()
lig_pdb = "\n".join([l for l in lig_str.splitlines() if l.startswith(("ATOM","HETATM"))])

view = py3Dmol.view(width=600, height=600)
view.addModel(rec_pdb, 'pdb')
view.setStyle({'cartoon': {'color':'spectrum'}})

view.addModel(lig_pdb, 'pdb')
view.setStyle({'stick': {'colorscheme':'greyCarbon'}})
view.setStyle(
    {'model': 1},
    {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.35}}
)


view.zoomTo()
view.show()
